## LSTM on MNIST — 逐行序列识别

### 实验思路

将 28×28 图片看作**28 个时间步**的序列（每步一行 28 像素），用双向 LSTM 逐行"阅读"后做分类。

### 为什么用 LSTM 处理图片？

```
     时间步1      时间步2           时间步28
    ┌────────┐  ┌────────┐       ┌────────┐
    │ 第 1 行 │  │ 第 2 行 │  ...  │ 第28行 │
    │ 28 像素 │  │ 28 像素 │       │ 28 像素 │
    └───┬────┘  └───┬────┘       └───┬────┘
        │   ┌──────┘                │
        ▼   ▼                       ▼
    ┌──────────────────────────────────┐
    │       双向 LSTM (2层×128)        │
    │  前向: 从左到右扫描               │
    │  后向: 从右到左扫描               │
    │  拼接: 256维隐状态               │
    └──────────────┬───────────────────┘
                   ▼
            FC(256→64→10)
```

### 三个门控机制

| 门控 | 作用 | 通俗理解 |
|------|------|----------|
| **遗忘门** | 决定丢弃哪些旧信息 | "这一行不重要，忘了" |
| **输入门** | 决定存储哪些新信息 | "这一行有关键笔画，记住" |
| **输出门** | 决定输出哪些信息 | "现在看到的信息中，哪些对分类有用" |

### 实验意义

- 补全 **MLP → CNN → RNN → Transformer** 四大基础架构图谱
- MNIST 数字的笔画确实有方向性（书写顺序），LSTM 可捕获这种时序模式
- 仅 ~85K 参数，是 MLP 的 1/6，极轻量

> 预期准确率：~98.5%（虽低于 CNN，但证明序列建模也能处理图像分类）

## 1. 导入依赖 & 数据加载

LSTM 输入格式为 `(B, seq_len, feat_dim) = (B, 28, 28)`——不需要通道维度。

In [ ]:
# [1.1 导入依赖]
# LSTM 将 28×28 视为 28 时间步 × 28 特征序列 | 双向扫描 | 仅 ~85K 参数
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
import warnings
warnings.filterwarnings('ignore')

# ==== 中文字体配置 ====
# matplotlib 默认不支持中文，需手动指定支持中文的字体
# 优先级：微软雅黑 > 黑体 > DejaVu Sans（英文回退）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 防止负号显示为方块

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')

In [ ]:
# [1.2 加载 MNIST 数据]
# LSTM 输入格式: (B, seq_len, feat_dim) = (B, 28, 28)
# 保持 (N, 28, 28) 不加通道维
mnist = fetch_openml(
    name='mnist_784', version=1, as_frame=False,
    cache=True, data_home='../data'
)
X = mnist.data.reshape(-1, 28, 28).astype(np.uint8)
y = mnist.target.astype(np.uint8)
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

# 归一化到 [0,1]，不展平，不加通道维
X_train_t = torch.tensor(X_train, dtype=torch.float32) / 255.0
X_test_t  = torch.tensor(X_test,  dtype=torch.float32) / 255.0
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# ======== 超参数集中修改区 ========
BATCH_SIZE = 128
# =================================

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=BATCH_SIZE, shuffle=False)

print(f'训练批次数: {len(train_loader)}, 测试批次数: {len(test_loader)}')
print(f'输入形状: {X_train_t.shape}')  # (60000, 28, 28) = (B, seq_len, feat_dim)

## 2. 可视化：把图片当序列看

热力图展示一张图 28 个时间步的像素值分布，直观理解 LSTM 的"视角"——逐行扫描图片。

In [ ]:
# [2. 序列可视化]
# 上方: 原图 | 下方: 逐行像素热力图 (28 时间步 × 28 特征)
sample_idx = 0
sample = X_train[sample_idx]
label = y_train[sample_idx]

fig, axes = plt.subplots(2, 1, figsize=(10, 5))

# 上方：原始图片
axes[0].imshow(sample, cmap='gray')
axes[0].set_title(f'原始图片 (标签={label})', fontsize=12)
axes[0].axis('off')

# 下方：28 行像素作为 28 个时间步的热力图
im = axes[1].imshow(sample.T, cmap='hot', aspect='auto')
axes[1].set_xlabel('时间步 (行号)')
axes[1].set_ylabel('像素位置')
axes[1].set_title('28 个时间步，每步 = 28 个像素值', fontsize=12)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 3. 定义 LSTM 模型

### 双向 LSTM 的原理

```
正向 LSTM: 第 1 行 → 第 2 行 → ... → 第 28 行
反向 LSTM: 第 28 行 → 第 27 行 → ... → 第 1 行
                    ↓
         拼接两个方向的最后隐状态
         (B, 128) + (B, 128) → (B, 256)
                    ↓
              FC(256→64→10)
```

### 关键参数说明

- `batch_first=True`：输入格式为 `(B, seq, feat)` 而非默认的 `(seq, B, feat)`
- `bidirectional=True`：正反两个方向扫描，隐状态维度翻倍
- `num_layers=2`：两层 LSTM 堆叠，第二层的输入是第一层的输出

In [ ]:
# [3. 定义 LSTM 模型]
class LSTMModel(nn.Module):
    """
    双向 LSTM + 全连接分类头

    输入: (B, 28, 28) — 28 个时间步，每步 28 维特征
    架构: LSTM(28→128, 2层, 双向) → concat(h_forward, h_backward) → FC(256→64→10)
    参数量: ~85K

    可调参数：
        input_dim:      每步的特征维度，MNIST 为 28（一行像素数）
        hidden_dim:     隐状态维度，默认 128
        num_layers:     LSTM 层数，默认 2
        bidirectional:  是否双向，默认 True
    """
    def __init__(self, input_dim=28, hidden_dim=128, num_layers=2,
                 num_classes=10, dropout=0.3, bidirectional=True):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,          # (B, seq, feat) 格式
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)

        # 双向 LSTM 隐状态拼接后维度翻倍
        fc_input_dim = hidden_dim * self.num_directions
        self.fc = nn.Sequential(
            nn.Linear(fc_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # lstm_out: (B, 28, hidden*directions) — 所有时间步的输出
        # h_n:      (layers*directions, B, hidden) — 最后时间步的隐状态
        lstm_out, (h_n, c_n) = self.lstm(x)

        if self.bidirectional:
            # 拼接最后时间步的正反向隐状态: (B, hidden*2)
            h_forward  = h_n[-2, :, :]  # 正向最后层
            h_backward = h_n[-1, :, :]  # 反向最后层
            h_last = torch.cat([h_forward, h_backward], dim=1)
        else:
            h_last = h_n[-1, :, :]

        out = self.dropout(h_last)
        return self.fc(out)


# 实例化模型
model = LSTMModel(input_dim=28, hidden_dim=128, num_layers=2,
                  num_classes=10, dropout=0.3, bidirectional=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')
print()
print(model)

## 4. 损失函数与优化器

Adam + ReduceLROnPlateau。LSTM 对学习率相对敏感，ReduceLROnPlateau 的自动衰减很有帮助。

In [ ]:
# [4. 损失函数 + 优化器 + 学习率调度器]

# ======== 超参数集中修改区 ========
LR = 0.001
WEIGHT_DECAY = 1e-4
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 5
# =================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE, verbose=True
)

print(f'优化器: Adam(lr={LR}, weight_decay={WEIGHT_DECAY})')
print(f'调度器: ReduceLROnPlateau(factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})')

## 5. 训练与评估函数

标准的 PyTorch 训练流程。

In [ ]:
# [5. 训练/评估函数]
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回 (avg_loss, accuracy)"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """评估模型，返回 (avg_loss, accuracy)"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

## 6. 启动训练

训练 50 个 epoch。自动保存最佳模型和周期性 checkpoint。

In [ ]:
# [6. 训练循环]

# ======== 超参数集中修改区 ========
EPOCHS = 50
CHECKPOINT_INTERVAL = 10
SAVE_DIR = '../models/lstm'
# =================================

history = {
    'train_loss': [], 'train_acc': [],
    'test_loss': [],  'test_acc': []
}

best_test_acc = 0.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    # ReduceLROnPlateau 根据验证损失调整
    scheduler.step(test_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    # 保存最优模型
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_epoch = epoch
        torch.save(model.state_dict(), f'{SAVE_DIR}/best.pth')

    # 定期保存检查点
    if epoch % CHECKPOINT_INTERVAL == 0:
        torch.save(model.state_dict(), f'{SAVE_DIR}/epoch_{epoch}.pth')
        print(f'  [checkpoint] 已保存: lstm_epoch_{epoch}.pth')

    # 每 5 个 epoch 或第 1 个 epoch 打印进度
    if epoch % 5 == 0 or epoch == 1:
        print(
            f'Epoch [{epoch:3d}/{EPOCHS}] '
            f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
            f'Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}'
        )

print(f'\n训练完成！最佳测试准确率: {best_test_acc:.4f} (Epoch {best_epoch})')

## 7. 训练曲线可视化

左侧 Loss 曲线观察收敛情况，右侧 Accuracy 曲线用虚线标出最佳准确率。

In [ ]:
# [7. 训练曲线可视化]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['test_loss'], label='Test Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('LSTM Loss 曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：Accuracy 曲线
axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['test_acc'], label='Test Acc')
axes[1].axhline(y=best_test_acc, color='r', linestyle='--', alpha=0.5,
                label=f'Best: {best_test_acc:.4f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('LSTM Accuracy 曲线')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 进一步学习

1. 将 `bidirectional` 改为 `False`，观察单向 LSTM 的准确率变化
2. 修改 `hidden_dim` 观察模型容量的影响
3. 尝试用 GRU 替换 LSTM（`nn.GRU`），对比参数数量和性能